In [ ]:
%pip -q install requests pandas numpy scikit-learn matplotlib

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
FDC_SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"
RETRIEVED_AT_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")

try:
    from google.colab import userdata
    API_KEY = userdata.get("FDC_API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    API_KEY = getpass("USDA FDC API key (input hidden): ").strip()
if not API_KEY:
    raise ValueError("กรุณาตั้ง FDC_API_KEY ใน Colab Secrets หรือกรอก API key")

print("Environment ready. API key is stored in memory and will not be displayed.")

In [ ]:
SEARCH_TERMS = [
    "chicken breast", "tofu", "brown rice", "egg", "salmon",
    "oats", "banana", "lentils", "spinach", "yogurt", "shrimp",
    "potato", "beans", "pork loin", "whole wheat bread",
    "sweet potato", "broccoli", "cod", "milk", "chickpeas",
]
DATA_TYPES = ["Foundation", "SR Legacy"]
PAGE_SIZE = 25
REQUEST_PAUSE_SECONDS = 0.15

nutrient_ids = {
    1008: "kcal",
    1003: "protein_g",
    1005: "carbs_g",
    1004: "fat_g",
    1093: "sodium_mg",
}

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=3))

def search_fdc(term):
    response = session.post(
        FDC_SEARCH_URL,
        params={"api_key": API_KEY},
        json={"query": term, "dataType": DATA_TYPES, "pageSize": PAGE_SIZE},
        timeout=45,
    )
    response.raise_for_status()
    return response.json().get("foods", [])

rows = []
query_errors = []
for index, term in enumerate(SEARCH_TERMS, start=1):
    try:
        foods = search_fdc(term)
        for food in foods:
            nutrient_values = {
                item.get("nutrientId"): item.get("value")
                for item in food.get("foodNutrients", [])
                if item.get("nutrientId") in nutrient_ids
            }
            row = {
                "fdc_id": food.get("fdcId"),
                "description": food.get("description", ""),
                "data_type": food.get("dataType", ""),
                "query_term": term,
                "retrieved_at_utc": RETRIEVED_AT_UTC,
            }
            row.update({column: nutrient_values.get(nutrient_id) for nutrient_id, column in nutrient_ids.items()})
            rows.append(row)
        print(f"[{index:02d}/{len(SEARCH_TERMS)}] {term}: {len(foods)} results")
    except requests.RequestException as exc:
        query_errors.append({"query_term": term, "error": str(exc)})
        print(f"[{index:02d}/{len(SEARCH_TERMS)}] {term}: request failed ({type(exc).__name__})")
    time.sleep(REQUEST_PAUSE_SECONDS)

raw_df = pd.DataFrame(rows)
if raw_df.empty:
    raise RuntimeError("ไม่พบข้อมูลจาก FDC API; ตรวจ API key, internet connection และลองอีกครั้ง")

print(f"\nRetrieved {len(raw_df):,} search rows; failed queries: {len(query_errors)}")

In [ ]:
display(raw_df.head(10))
print("Data types:")
display(raw_df["data_type"].value_counts(dropna=False).rename_axis("data_type").to_frame("rows"))

In [ ]:
required_columns = ["fdc_id", "description", "data_type", "kcal", "protein_g", "sodium_mg"]
missing_columns = sorted(set(required_columns) - set(raw_df.columns))
if missing_columns:
    raise KeyError(f"FDC response missing expected fields: {missing_columns}")

numeric_columns = ["kcal", "protein_g", "carbs_g", "fat_g", "sodium_mg"]
for column in numeric_columns:
    raw_df[column] = pd.to_numeric(raw_df[column], errors="coerce")

before_dedup = len(raw_df)
df = raw_df.dropna(subset=["fdc_id"]).drop_duplicates(subset=["fdc_id"]).copy()
deduplicated_count = before_dedup - len(df)
df = df.dropna(subset=["kcal", "protein_g", "sodium_mg"])
df = df[(df["kcal"] > 0) & (df["protein_g"] >= 0) & (df["sodium_mg"] >= 0)]
df = df.reset_index(drop=True)

print(f"Duplicate records removed: {deduplicated_count:,}")
print(f"Valid unique foods: {len(df):,}")
if len(df) < 30:
    raise RuntimeError(
        f"เหลือข้อมูลเพียง {len(df)} รายการ (ต้องการอย่างน้อย 30) "
        "ให้เพิ่มคำค้น ตรวจสอบ API key หรือดู response จาก FDC"
    )

In [ ]:
data_dictionary = pd.DataFrame([
    ("fdc_id", "รหัสรายการอาหารใน FoodData Central"),
    ("description", "ชื่ออาหารตาม FDC; เก็บเป็นข้อความต้นทาง"),
    ("data_type", "Foundation หรือ SR Legacy"),
    ("query_term", "คำค้นที่พบ record นี้ครั้งแรก"),
    ("kcal", "พลังงานตามค่าที่ส่งคืนจาก FDC"),
    ("protein_g", "โปรตีน หน่วยกรัม"),
    ("carbs_g", "คาร์โบไฮเดรต หน่วยกรัม"),
    ("fat_g", "ไขมัน หน่วยกรัม"),
    ("sodium_mg", "โซเดียม หน่วยมิลลิกรัม"),
    ("retrieved_at_utc", "เวลา snapshot ที่ดึงข้อมูล"),
], columns=["column", "description"])
display(data_dictionary)
display(df[numeric_columns].describe().T.round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, column, title, color in [
    (axes[0, 0], "kcal", "Energy (kcal)", "#4f7d55"),
    (axes[0, 1], "protein_g", "Protein (g)", "#ce8d45"),
    (axes[1, 0], "sodium_mg", "Sodium (mg)", "#c76e60"),
    (axes[1, 1], "fat_g", "Fat (g)", "#7188a3"),
]:
    ax.hist(df[column].dropna(), bins=20, color=color, edgecolor="white")
    ax.set_title(title)
    ax.set_ylabel("Number of FDC foods")
    ax.grid(axis="y", alpha=0.2)
fig.suptitle("USDA FoodData Central snapshot: nutrient distributions", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
snapshot_path = Path("/content/usda_fooddata_central_sample.csv")
df.to_csv(snapshot_path, index=False)
print(f"Saved {snapshot_path.name} ({snapshot_path.stat().st_size:,} bytes)")
display(df[["fdc_id", "description", "data_type", *numeric_columns]].head(12))

In [ ]:
FEATURE_NAMES = [
    "budget_fit", "protein", "low_sodium",
    "preference_match", "calorie_fit", "popularity",
]
PROXY_WEIGHTS = np.array([0.15, 0.35, 0.25, 0.15, 0.10, 0.00], dtype=float)
TARGET_CALORIES = 500.0

train_foods, test_foods = train_test_split(
    df, test_size=0.20, random_state=RANDOM_STATE,
)

def make_features(frame, seed):
    rng = np.random.default_rng(seed)
    kcal = frame["kcal"].to_numpy(dtype=float)
    protein_g = frame["protein_g"].to_numpy(dtype=float)
    sodium_mg = frame["sodium_mg"].to_numpy(dtype=float)
    return pd.DataFrame({
        "budget_fit": rng.uniform(0.25, 1.0, len(frame)),
        "protein": np.clip(protein_g / 35.0, 0.0, 1.0),
        "low_sodium": np.clip(1.0 - sodium_mg / 1800.0, 0.0, 1.0),
        "preference_match": rng.uniform(0.0, 1.0, len(frame)),
        "calorie_fit": np.clip(1.0 - np.abs(kcal - TARGET_CALORIES) / TARGET_CALORIES, 0.0, 1.0),
        "popularity": np.full(len(frame), 0.5),
    }, columns=FEATURE_NAMES, index=frame.index)

def make_proxy_target(features):
    return features.to_numpy() @ PROXY_WEIGHTS

X_train = make_features(train_foods, RANDOM_STATE)
X_test = make_features(test_foods, RANDOM_STATE + 1)
y_train = make_proxy_target(X_train)
y_test = make_proxy_target(X_test)

print(f"Train FDC records: {len(X_train):,} | test FDC records: {len(X_test):,}")
display(X_train.head())

In [ ]:
proxy_definition = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "proxy_weight": PROXY_WEIGHTS,
    "interpretation": [
        "simulated fit to budget (FDC has no price)",
        "protein grams normalized by 35 g",
        "lower sodium receives higher score",
        "simulated match to user preference (no FDC feedback)",
        "closeness to a 500 kcal target",
        "constant neutral value; no FDC popularity data",
    ],
})
display(proxy_definition)

In [ ]:
model = make_pipeline(StandardScaler(), Ridge(alpha=0.05))
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
baseline_pred = make_proxy_target(X_test)

def ndcg_at_k(y_true, y_score, k=10):
    y_true = np.asarray(y_true, dtype=float)
    y_score = np.asarray(y_score, dtype=float)
    k = min(k, len(y_true))
    if k == 0:
        return float("nan")
    order = np.argsort(y_score)[::-1][:k]
    ideal = np.argsort(y_true)[::-1][:k]
    discounts = np.log2(np.arange(2, k + 2))
    dcg = np.sum((2 ** y_true[order] - 1) / discounts)
    idcg = np.sum((2 ** y_true[ideal] - 1) / discounts)
    return float(dcg / idcg) if idcg else 0.0

try:
    from scipy.stats import spearmanr
    spearman = float(spearmanr(y_test, y_pred).statistic)
except Exception:
    spearman = float("nan")

def metric_row(name, truth, prediction):
    return {
        "model": name,
        "MAE": mean_absolute_error(truth, prediction),
        "RMSE": mean_squared_error(truth, prediction) ** 0.5,
        "R²": r2_score(truth, prediction),
        "NDCG@10": ndcg_at_k(truth, prediction, 10),
    }

metrics_df = pd.DataFrame([
    metric_row("Ridge (trained)", y_test, y_pred),
    metric_row("Proxy formula baseline", y_test, baseline_pred),
])
display(metrics_df.round(4))
print(f"Spearman rank correlation (trained model vs proxy): {spearman:.4f}")

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_mae = -cross_val_score(
    model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error",
)
print("5-fold CV MAE on training proxy labels:")
print(f"mean={cv_mae.mean():.4f}, std={cv_mae.std():.4f}")

In [ ]:
ridge = model.named_steps["ridge"]
scaler = model.named_steps["standardscaler"]
export_weights = ridge.coef_ / scaler.scale_
export_bias = ridge.intercept_ - np.sum(ridge.coef_ * scaler.mean_ / scaler.scale_)

coefficient_df = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "proxy_weight": PROXY_WEIGHTS,
    "ridge_weight": export_weights,
}).sort_values("ridge_weight", ascending=False)
display(coefficient_df.round(4))

ax = coefficient_df.set_index("feature")[["proxy_weight", "ridge_weight"]].plot(
    kind="bar", figsize=(10, 4), color=["#c9a66b", "#426b4f"],
)
ax.set_title("Declared proxy weights vs learned Ridge weights")
ax.set_ylabel("Linear coefficient")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
test_results = test_foods[["fdc_id", "description", "data_type", "kcal", "protein_g", "sodium_mg"]].copy()
test_results["proxy_target"] = y_test
test_results["ridge_score"] = y_pred
test_results["baseline_score"] = baseline_pred
print("Top 10 held-out FDC foods by learned proxy score:")
display(test_results.sort_values("ridge_score", ascending=False).head(10).reset_index(drop=True))

In [ ]:
artifact = {
    "version": "1.1-usda-proxy",
    "features": FEATURE_NAMES,
    "weights": [float(value) for value in export_weights],
    "bias": float(export_bias),
    "trained": True,
    "note": "Ridge model trained on USDA FDC nutrient records with a declared weak-supervision proxy; not user ratings.",
}

output_dir = Path("/content")
model_path = output_dir / "model.json"
model_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")

print("Feature contract:", artifact["features"])
print("Weights:", [round(value, 4) for value in artifact["weights"]])
print("Bias:", round(artifact["bias"], 4))
print("Trained:", artifact["trained"])
print("\nSaved:", model_path)
print(model_path.read_text(encoding="utf-8"))

In [ ]:
from google.colab import files

files.download(str(model_path))
files.download(str(snapshot_path))